# 第35课 · 单位圆上的秒针——用欧拉公式（Euler's formula）亲手铸出 FFT 的旋转因子（twiddle factor）

**今日目标**：用 `cos θ + i·sin θ` 拼出 `euler(θ)`，再导出 `twiddle(k, n, N) = euler(-2πkn/N)`。

> **复习 L06 + DSP 约定**：欧拉公式 L06 已讲。本课在 DFT 索引 k,n,N 下铸成可调用函数；**负号**跟 DFT 定义一致，勿随意改。不必第三遍完整证明欧拉。

L37–L39 重写 FFT 时每一行都调用 `twiddle`；今天写好，后面直接复用。

← **上一课**　[L34 · Nyquist 定理与混叠](L34_aliasing.ipynb)

> 上节课学习了 **Nyquist 定理与混叠**：6kHz 正弦波被 8kHz 采样后会变成什么。  
> 本课将探讨 **欧拉公式遇见 FFT**。

## 学习目标

完成本课后，你应该能够：

1. **实现 `euler(theta)`**：用 `np.cos` 和 `np.sin` 从第一原理构造 `e^{iθ}`，不依赖 `np.exp`。
2. **实现 `twiddle(k, n, N)`**：调用 `euler` 计算 FFT 旋转因子 `e^{-2πikn/N}`，说明负号的物理含义（顺时针 / 分析方向）。
3. **通过断言验收**：让验证 cell 的所有 assert 全部通过，包括 `twiddle(1,2,8) == -1j` 这个非平凡 case。
4. **连接 Aurora 源码**：能在 `src/aurora/audio/transforms.py` 中找到使用等效旋转因子矩阵的代码位置。

## 复习桥 · L07 方波 = 谐波叠加（2 分钟）

L07 已证：周期波形可写成正弦波的加权和——方波 ≈ 奇次谐波 `sin(2πkt)/k` 的叠加。

DFT/FFT 做的是**逆问题**：给定一段信号，问「每个频率分量有多大？」
今天写的 `twiddle(k, n, N)` 就是离散时间第 `n` 步、频率 `k` 上的旋转位置。

带着这个对照进入欧拉公式：复指数（complex exponential）不是新东西，是把 sin/cos 合并成单位圆上的坐标。


## 本课剧情：旋转的密码

FFT 的核心公式里有个神秘指数：`e^{-2πikn/N}`。它不是普通的实数指数，而是一个**在单位圆上旋转的复数**。

**欧拉公式（Euler's formula）**：
$$e^{i\theta} = \cos\theta + i\sin\theta$$

直觉：想象一只手表的秒针，固定在单位圆上。角度 θ 告诉它转到哪个位置——横坐标是 cos θ，纵坐标是 sin θ。把这两个坐标合并成一个复数，就是 `e^{iθ}`。

**为什么这个公式成立？**（不需要背，了解思路就行）

从泰勒级数的角度，我们知道：
$$e^x = 1 + x + \frac{x^2}{2!} + \frac{x^3}{3!} + \ldots$$

如果让 `x = iθ`（虚数），代入得：
$$e^{i\theta} = 1 + i\theta + \frac{(i\theta)^2}{2!} + \frac{(i\theta)^3}{3!} + \ldots$$

注意 `i² = -1`，`i³ = -i`，`i⁴ = 1`…… 这样就会出现奇偶项分离：
- 偶数项（1, -θ²/2!, θ⁴/4!, …）恰好是 `cos θ` 的泰勒级数
- 奇数项（iθ, -iθ³/3!, …）恰好是 `i·sin θ` 的泰勒级数

所以 `e^{iθ} = cos θ + i·sin θ` 不是凭空定义，而是指数函数和三角函数的深层联系。

**复数乘法的几何性质（极坐标视角）**：

两个复数相乘时，如果用极坐标表示（模 r、辐角 θ）：
$$(r_1 e^{i\theta_1}) \times (r_2 e^{i\theta_2}) = (r_1 r_2) e^{i(\theta_1 + \theta_2)}$$

意思是：**模相乘，辐角相加。** 这就是为什么乘以单位圆上的点（模为1的复数）只会旋转而不会缩放——模相乘的结果还是 1。

**为什么 FFT 需要它？**

DFT（离散傅里叶变换）的第 k 个输出：

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i k n/N}$$

每个 `e^{-2πikn/N}` 都是一个**旋转因子（twiddle factor）**：秒针以频率 k 旋转，在第 n 步时到达的位置。把信号 x[n] 乘以这个旋转因子，相当于"问 x[n] 在这个频率上的投影有多大"。

**"投影"到底是什么意思？（生活类比）**

想象你在一间暗屋子里，手里举着一支手电筒，沿着某个固定方向照向一个正在来回摆动的小球——手电筒的方向就是"频率 k 对应的旋转方向"。小球在这个方向上投下的影子有多长，就是"投影"：如果小球摆动的方向恰好和光线方向一致，影子会很长（说明这个频率上的能量大）；如果两者垂直，影子几乎消失为一个点（说明这个频率上几乎没有能量）。

DFT 做的事情，就是对每一个候选频率 k，"打一束特定方向、并且随时间旋转的光"（也就是乘以旋转因子 `e^{-2πikn/N}`），然后把所有采样点的"影子"加起来（对 n 求和，也就是公式里的 Σ）。如果信号里真的含有频率 k 的成分，一路上的影子会朝着差不多的方向持续叠加，最后加出一个绝对值很大的复数；如果信号里没有这个频率，正负方向的影子会互相抵消，加出一个接近 0 的结果。这就是"投影有多大"的直观含义——**累加得越多，说明信号在这个频率上"藏得越深"**。

本节实现两个函数：
- `euler(theta)` → `cos(theta) + i·sin(theta)` （欧拉公式）
- `twiddle(k, n, N)` → `e^{-2πi·k·n/N}` （FFT 旋转因子）

## 1. 先把复数当成平面坐标

- 复数 `a + b·i` 对应平面上的点 `(a, b)`。
- 实部 `a` 是横坐标，虚部 `b` 是纵坐标。
- 乘以 `i` 会把点逆时针旋转 90°。
- `e^{iθ}` 是单位圆（unit circle）上角度为 `θ` 的点，也就是 `(cos θ, sin θ)`。

**为什么单位圆上角度为 θ 的点坐标就是 (cos θ, sin θ)？**

想象一条从原点到单位圆周上某点的线段，它与正 x 轴夹角为 θ。从该点向下（x轴）和向左（y轴）作垂线，两条垂线的长度分别就是 cos θ 和 sin θ——这就是直角三角形中三角函数的定义。所以把这个点写成复数，自然就是 `cos θ + i·sin θ`。

**复数乘法为什么会旋转？**

如果两个复数写成极坐标形式：
- `z₁ = r₁·e^{iθ₁}` 表示模为 r₁、辐角为 θ₁ 的复数
- `z₂ = r₂·e^{iθ₂}` 表示模为 r₂、辐角为 θ₂ 的复数

它们相乘：
$$z_1 \times z_2 = r_1 e^{i\theta_1} \times r_2 e^{i\theta_2} = (r_1 r_2) \cdot e^{i(\theta_1 + \theta_2)}$$

结果是：模变成 `r₁×r₂`，辐角变成 `θ₁+θ₂`。用通俗话说就是**"旋转 θ₂，再缩放 r₂ 倍"**。

特别地，如果 z₂ 在单位圆上（r₂ = 1），那就**只旋转不缩放**。这就是为什么 FFT 用单位圆上的复数（旋转因子）去乘信号——它只改变相位，不改变幅度。

**为什么用 cos+i·sin 手动拼，而不是直接写 `np.exp(1j*theta)`？** 两种写法数值完全相同，但手动拼出实部和虚部可以让你亲眼看到欧拉公式成立——而不只是把它当成黑盒函数调用。L37-L39 重写 FFT 时，每对 `(k, n)` 调用一次 `twiddle`，N² 次调用后拼出完整频谱；今天理解底层构造，下周才能看懂每行矩阵在做什么。

## 1.1 复数乘法先看 90° 旋转

先不进入公式，只看 `z * 1j`。虚数单位 `1j` 在单位圆上对应辐角 π/2（90°）。根据极坐标乘法法则"辐角相加"，任何复数乘以 `1j` 就相当于它的辐角加上 π/2，也就是逆时针旋转 90°。

每乘一次 `1j`，点就绕原点逆时针转 90°。看下面代码验证：

### 🤔 为什么要用虚数，而不是 (cos θ, sin θ) 向量对？

你可能会问：既然 cos θ 和 sin θ 就能表示单位圆上的点，为什么一定要用复数 `cos θ + i·sin θ`？实数对 `(cos θ, sin θ)` 不也可以吗？

**答案是：虚数让旋转变成乘法。**

- 用**向量对**：旋转需要矩阵 — 如果要把点 `(x, y)` 旋转 θ 角度，需要用旋转矩阵，这是一个 2×2 的操作。
- 用**复数**：旋转只需乘法 — 一个点 `z = x + iy` 乘以 `e^{iθ}` 就自动旋转 θ 角度了，无需矩阵。

**虚数单位 `i` 的几何含义就是"逆时针旋转 90°"。** 看看：
- `1 × i = i`（点 `(1,0)` 乘以 i，变成 `(0,1)`，旋转了 90°）
- `i × i = -1`（点 `(0,1)` 再乘以 i，变成 `(-1,0)`，再旋转 90°，共 180°）

这就是为什么 FFT 用复数而不是向量对：**旋转要反复进行，用乘法远比用矩阵简洁。**

In [1]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [2]:
import numpy as np

z = 1 + 0j
for step in range(5):
    print(f'step={step}  z={z.real:+.0f}{z.imag:+.0f}j')
    z = z * 1j


step=0  z=+1+0j
step=1  z=+0+1j
step=2  z=-1+0j
step=3  z=-0-1j
step=4  z=+1-0j


## 实验入口：观察角度序列

这组实验用均匀分布在 [0, 2π) 的角度作为输入，同时对照 cos、sin 和复数坐标，确认三者始终指向单位圆上同一个点。

In [3]:
import numpy as np
print('就绪；虚数单位示例:', 1j * 1j)  # = (-1+0j)

就绪；虚数单位示例: (-1+0j)


## 动手观察：从采样点到角度序列

下面的 cell 把一个正弦波拆开给你看：采样点编号 `n` → 时刻 `t` → 圆上角度 `angle = 2π·f·t` → 波形值 `sin(angle)`。调整 `freq` 或 `sample_rate`，观察角度序列如何变化——这些角度正是稍后要送进欧拉公式的输入。

In [4]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('采样点编号 n =', n)
print('时间轴 t =', np.round(t, 3))
print('角度 angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


N = 8
采样点编号 n = [0 1 2 3 4 5 6 7]
时间轴 t = [0.    0.125 0.25  0.375 0.5   0.625 0.75  0.875]
角度 angle = [ 0.     1.571  3.142  4.712  6.283  7.854  9.425 10.996]
sin(angle) = [ 0.  1.  0. -1. -0.  1.  0. -1.]


## 代码实验：角度、三角函数和复数坐标对齐

并排打印同一批角度的 cos、sin 和 exp(1j·θ) 三列，验证 exp(1j·θ) 的实部等于 cos θ、虚部等于 sin θ。

In [5]:
import numpy as np

angles = np.linspace(0, 2*np.pi, 9)
for theta in angles:
    z = np.exp(1j * theta)
    print(f'theta={theta:5.2f} | cos={z.real:6.3f} | sin={z.imag:6.3f} | z={z.real:6.3f}{z.imag:+6.3f}j')


theta= 0.00 | cos= 1.000 | sin= 0.000 | z= 1.000+0.000j
theta= 0.79 | cos= 0.707 | sin= 0.707 | z= 0.707+0.707j
theta= 1.57 | cos= 0.000 | sin= 1.000 | z= 0.000+1.000j
theta= 2.36 | cos=-0.707 | sin= 0.707 | z=-0.707+0.707j
theta= 3.14 | cos=-1.000 | sin= 0.000 | z=-1.000+0.000j
theta= 3.93 | cos=-0.707 | sin=-0.707 | z=-0.707-0.707j
theta= 4.71 | cos=-0.000 | sin=-1.000 | z=-0.000-1.000j
theta= 5.50 | cos= 0.707 | sin=-0.707 | z= 0.707-0.707j
theta= 6.28 | cos= 1.000 | sin=-0.000 | z= 1.000-0.000j


## 2. ✏️ 实现 `euler(theta)`

**手算三个特殊角度**（单位圆上的关键点）：

| theta (弧度) | cos θ | sin θ | e^{iθ} |
|---|---|---|---|
| 0 | 1 | 0 | **1+0j** |
| π/2 ≈ 1.5708 | 0 | 1 | **0+1j**（逆时针转90°） |
| π ≈ 3.1416 | -1 | 0 | **-1+0j**（转180°，秒针指左） |
| 3π/2 | 0 | -1 | **0-1j** |
| 2π | 1 | 0 | **1+0j**（转一圈，回原点） |

**关键性质**：|e^{iθ}| = √(cos²θ + sin²θ) = **1**（永远在单位圆上）

> **实现提示**：`return np.cos(theta) + 1j * np.sin(theta)`

### 写代码前，先把变量表补完整

写 `euler` 前明确三件事：
- 输入：`theta`（弧度，标量或数组）
- 关键步骤：计算 `np.cos(theta)` 作实部，`np.sin(theta)` 作虚部，相加得复数
- 返回：复数（或复数数组），模为 1，辐角为 theta

**`1j` 和数组相乘，NumPy 会怎么处理？**

`1j` 是一个复数标量（实部 0、虚部 1）。当你写 `1j * np.sin(theta)`，而 `theta` 是一个数组（比如 9 个角度）时，NumPy 会触发**广播（broadcasting）**：标量 `1j` 会被"复制"到数组的每一个位置，逐元素相乘，得到一个新的复数数组——第 i 个元素就是 `1j * sin(theta[i])`。

再加上 `np.cos(theta)`（一个实数数组），NumPy 会自动把它提升为复数类型，逐元素相加。最终结果是一个 `dtype=complex128` 的数组，每个元素恰好是 `cos(theta[i]) + i·sin(theta[i])`。

也就是说：不管 `theta` 是一个数字还是一整个数组，`np.cos(theta) + 1j * np.sin(theta)` 这一行代码写法完全不变——这正是 NumPy 广播机制带来的好处，不需要手写循环。下面这个小实验可以亲眼验证这个广播过程：

In [6]:
# 广播小实验：1j 乘以数组会发生什么？
import numpy as np

demo_theta = np.array([0, np.pi/2, np.pi])   # 3 个角度组成的数组
demo_sin = np.sin(demo_theta)
demo_result = 1j * demo_sin                   # 标量 1j 广播到每个元素

print('demo_theta    =', demo_theta)
print('demo_sin      =', demo_sin)
print('1j * demo_sin =', demo_result)
print('结果类型:', demo_result.dtype, '| 形状:', demo_result.shape)

# 加上 cos 部分，验证和 euler 公式的结构完全一致
demo_cos = np.cos(demo_theta)
demo_full = demo_cos + demo_result
print('\ncos(theta) + 1j*sin(theta) =', demo_full)
print('✅ 标量 1j 自动广播到数组每一个位置，theta 是数组时代码写法不用变。')


demo_theta    = [0.         1.57079633 3.14159265]
demo_sin      = [0.0000000e+00 1.0000000e+00 1.2246468e-16]
1j * demo_sin = [0.+0.0000000e+00j 0.+1.0000000e+00j 0.+1.2246468e-16j]
结果类型: complex128 | 形状: (3,)

cos(theta) + 1j*sin(theta) = [ 1.000000e+00+0.0000000e+00j  6.123234e-17+1.0000000e+00j
 -1.000000e+00+1.2246468e-16j]
✅ 标量 1j 自动广播到数组每一个位置，theta 是数组时代码写法不用变。


In [7]:
def euler(theta):
    # ✏️ TODO: 返回 cosθ + i·sinθ
    raise NotImplementedError("TODO: 实现 euler(theta) — 返回 cos(theta) + 1j*sin(theta)")


## 3. 从单位圆到 FFT 旋转因子

`e^{iθ}` 表示“转到角度 θ 的位置”。FFT 需要的是一组规则旋转：

- `N`：一圈分成多少份
- `n`：当前采样点编号
- `k`：当前频率编号
- `-2π·k·n/N`：这个频率在这个采样点要转到的角度

所以 `twiddle(k, n, N)` 只是把这个角度送进欧拉公式。


### 3.1 ✏️ 任务二：FFT 旋转因子 `twiddle`

`W_N^{kn} = e^{-2πi·k·n/N}`

**推理路线**：
1. 旋转因子的角度是 `-2π·k·n/N`；**负号为什么必须是负的？**
   - 从共轭的角度：DFT 分析的是信号的**频率成分**，相当于在复平面上问"信号在 e^{+iωt} 这个方向上的投影有多大？"。为了提取正频率，我们用它的共轭 e^{-iωt} 来乘，这样同向的成分会加强，异向的会抵消。
   - 从数学习惯的角度：DFT 的逆变换（IDFT）用的是 `e^{+2πikn/N}`，这样才能互为逆运算。正变换用负号，逆变换用正号，形成对称。
   - 从旋转方向的角度：负号对应**顺时针旋转**（在通常的数学坐标系中，逆时针是正方向）。虽然这是个习惯约定，但一旦选定，整个 FFT 生态都基于这个方向。
2. 把这个角度送入 `euler(theta)` 即可，不需要重新推导 cos/sin。
3. 优先调用 `euler` 而不是 `np.exp`：如果要替换底层实现，改一处 `euler` 就能全局生效。

**参考输入输出**：`twiddle(k=1, n=2, N=8)` = `euler(-π/2)` = 0-1j

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。

In [8]:
def twiddle(k, n, N):
    # ✏️ TODO: 返回 e^{-2πi·k·n/N}
    raise NotImplementedError("TODO: 实现 twiddle(k, n, N) — 返回 euler(-2*np.pi*k*n/N)")


## 4. 检查 + 打印单位圆上的点

In [ ]:
theta = np.linspace(0, 2*np.pi, 9)   # 0..360°, 9 个点
z = euler(theta)
assert np.max(np.abs(z - np.exp(1j*theta))) < 1e-12, 'euler 应等于 np.exp(1j*theta)'
assert np.allclose(np.abs(z), 1.0), 'e^{iθ} 模长恒为 1（在单位圆上）'
assert abs(twiddle(0, 0, 8) - 1) < 1e-12, 'k·n=0 时旋转因子应为 1'
assert abs(twiddle(1, 2, 8) - (0-1j)) < 1e-12, 'twiddle(1,2,8) 应为 -1j（非平凡验证）'
for ang, val in zip(np.degrees(theta), z):
    print(f'{ang:6.0f}° | {val.real:+.3f} {val.imag:+.3f}i')
print('\n✅ 复指数=旋转，twiddle=FFT 里的旋转因子。')

### 补充：为什么 `twiddle(1, 2, 8) = -1j` 被称为"非平凡验证"？

上面的断言里有两行长得很像的检查：

```python
assert abs(twiddle(0, 0, 8) - 1) < 1e-12, 'k·n=0 时旋转因子应为 1'
assert abs(twiddle(1, 2, 8) - (0-1j)) < 1e-12, 'twiddle(1,2,8) 应为 -1j（非平凡验证）'
```

第一行其实很"平凡"：角度是 `-2π·0·0/8 = 0`，只要代码没有低级错误，`euler(0)` 几乎肯定等于 1（不管有没有理解欧拉公式，随便写个占位实现都很容易凑巧通过）。这一行测的只是"没有明显写崩"，测不出你是不是真的理解了公式的结构。

第二行 `twiddle(1, 2, 8) = -1j` 才是真正有信息量的测试，原因有三点：

1. **它恰好落在 90° 的整数倍上**：角度 = `-2π·1·2/8 = -π/2`（也就是 -90°）。这类角度的 cos/sin 精确值是已知的（0、+1、-1 这几个"整齐"的数），既不需要计算器，又能同时检验 cos 和 sin 两个分支有没有都算对——`cos(-π/2)=0` 且 `sin(-π/2)=-1`，两者都要对，才能拼出 `0 + (-1)i = -1j`。
2. **它能揪出符号写反、顺序算错这类隐蔽错误**：如果实现里把负号丢了、或者把 `k*n` 和 `N` 的位置搞反了，用一个随机角度去测，算出来的数字"看起来也像那么回事"，容易蒙混过关；但用 `-π/2` 这种关键角度去测，一旦算错，结果会明显偏离 `-1j`（比如变成 `+1j` 或 `1+0j`），错误立刻暴露。
3. **它是 FFT 蝶形运算里最常见的"标准积木"之一**：90° 整数倍的旋转因子（1、-1j、-1、1j）在基-2 FFT 的蝶形运算中反复出现。今天先把这个特例验证清楚，能给 L37-L39 手写 FFT 省下不少调试时间。

一句话总结：**`twiddle(0,0,8)=1` 只能测"没写崩"；`twiddle(1,2,8)=-1j` 测的是"欧拉公式和负号方向是不是真的用对了"——这就是它被称为"非平凡验证"的原因。**

## 5. 画出单位圆

In [ ]:
import matplotlib.pyplot as plt
fine = euler(np.linspace(0, 2*np.pi, 200))
plt.figure(figsize=(4.5, 4.5))
plt.plot(fine.real, fine.imag, '-', lw=1)
plt.plot(z.real, z.imag, 'o')
plt.gca().set_aspect('equal')
plt.title('e^{iθ} traces the unit circle')
plt.xlabel('Re'); plt.ylabel('Im'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 📊 图示：aviz.twiddles(8) 是什么？

下面这行代码会绘制 FFT 的旋转因子矩阵可视化——展示所有的 (k, n) 对组合出的 8×8 个旋转因子：

- **行索引**：频率 k (从 0 到 7)
- **列索引**：时间 n (从 0 到 7)
- **每个格子的颜色**：对应 `twiddle(k, n, 8)` 这个复数在单位圆上的位置（用颜色和方向表示）

这幅图会直观显示 FFT 矩阵中"每个旋转因子在单位圆上的位置"。你会看到：
- k=0 行（DC分量）全是 1，因为 `twiddle(0, n, 8) = e^0 = 1`
- 其他行呈周期性规律，按 k 的大小旋转速度递增

这就是下节课（L37-L39）实现 FFT 时要反复调用的"积木"。

**🎉 完成后**：`git commit -m 'learn: L35 euler'`（L37-L39 重写 FFT 全靠今天）

## 🎨 图示：8 个旋转因子 = FFT 的积木

In [ ]:
import aurora.aviz as aviz; aviz.style()
aviz.twiddles(8);

In [ ]:
# ✏️ 对比：np.exp(1j*theta) vs 手动拼 cos+sin
# 两种写法在数值上完全相同，但哪种更"透明"？

import numpy as np

print("验证：两种写法数值完全相同\n")
angles = np.linspace(0, 2*np.pi, 5)
for theta in angles:
    z_exp = np.exp(1j*theta)
    z_manual = np.cos(theta) + 1j*np.sin(theta)
    diff = np.abs(z_exp - z_manual)
    print(f'theta={theta:.2f} | exp={z_exp:.2f} | cos+i sin={z_manual:.2f} | diff={diff:.1e}')

print("\n✅ 差值在浮点精度范围内（< 1e-15），两种写法等价。")
print("\n为什么还要手动拼 cos+sin？")
print("- 用 np.exp(1j*theta)：简洁，但黑盒，看不出欧拉公式的结构")
print("- 手动拼 cos+sin：让实部和虚部显式出现，证明了 e^{iθ} = cos θ + i·sin θ")
print("- 在代码审阅和教学中，后者更能体现'为什么'的理解，而不只是'用了什么'。")

## 参数实验：只改一个旋钮

计算 `np.array([twiddle(1, n, 8) for n in range(8)])` 得到 8 个旋转因子，打印它们的相位（角度），确认分布为 0°, -45°, -90°, -135°, 180°, 135°, 90°, 45°——八个点均匀排列在单位圆上。

把 `N` 从 8 改成 16：相邻旋转因子的角度间距缩小一半（-22.5°），单位圆上的点密度翻倍。把 `k` 从 1 改成 3：每步旋转角度变为 -135°，FFT 频率编号直接控制旋转速度。

### 补充：twiddle(k, n, N) 中的参数含义

在 DFT 公式 `X[k] = Σ x[n] · e^{-2πikn/N}` 中，旋转因子 `e^{-2πikn/N}` 的三个参数各司其职：

| 参数 | 范围 | 含义 | 在旋转中的角色 |
|---|---|---|---|
| **k** | 0 到 N-1 | **频率编号**：在做频率分析时，第几个频率成分（0 Hz, Δf, 2Δf, …） | 控制**旋转速度**：k 越大，旋转因子转得越快 |
| **n** | 0 到 N-1 | **时间采样点编号**：第几个采样点（t=0, Δt, 2Δt, …） | 控制**旋转步数**：n 越大，已经转过的圈数越多 |
| **N** | 常数 | **总采样数**：一段信号包含多少个样本 | 控制**圆周分度**：N 越大，一圈分成的份数越多，相邻k的角频率间距越小 |

**数值角度 = -2π·k·n/N**：
- 分子 `k·n` 是两个小指标的乘积，每组 (k, n) 对应不同的角度
- 分母 `N` 是一圈分成 N 份，所以 `-2π/N` 是基本角度间隔
- 负号是习惯约定（见下文）

**举例**：假设 N=8（8个采样点），k=1（第1个频率）：
- n=0: 角度 = -2π·1·0/8 = 0°（秒针指向12点）
- n=1: 角度 = -2π·1·1/8 = -45°（顺时针转45°）
- n=2: 角度 = -2π·1·2/8 = -90°（顺时针转90°）
- ……
- n=7: 角度 = -2π·1·7/8 = -315°（顺时针转315°，等于逆时针转45°）

下一段实验会改变这三个参数，观察旋转因子如何变化。

In [ ]:
# ✏️ 参数实验：只改一个旋钮
import numpy as np

# --- 实验 A：固定 k=1，N=8，遍历 n=0..7 ---
N = 8
factors = np.array([twiddle(1, n, N) for n in range(N)])
phases_deg = np.degrees(np.angle(factors))
print('N=8, k=1 时各旋转因子的相位（度）：')
for n, (f, p) in enumerate(zip(factors, phases_deg)):
    print(f'  n={n}: {f.real:+.3f}{f.imag:+.3f}j  ∠{p:+.1f}°')

# --- 实验 B：改 N=16，观察角度间距缩小一半 ---
N2 = 16
factors2 = np.array([twiddle(1, n, N2) for n in range(N2)])
phases2 = np.degrees(np.angle(factors2))
print(f'\nN={N2}, k=1 时相邻旋转因子角度间距：{abs(phases2[1]-phases2[0]):.1f}°（N=8 时为 45°，缩小了一半）')

## 本课收束

现在可以调用 `euler(theta)` 生成单位圆上任意角度的复数，也可以调用 `twiddle(k, n, N)` 计算 FFT 所需的旋转因子。这两个函数对应 `src/aurora/audio/transforms.py` 里 L37-L39 DFT 实现的底层调用。下一课（L36）会用窗函数对信号加权，处理后的信号最终会送进今天写好的 `twiddle`。

本节与 `notebooks/1_complex_trig/L06_euler.ipynb` 覆盖同一个公式，**分工不同**：

| L06 | L35 |
|---|---|
| 重点：**欧拉公式的数学直觉** | 重点：**欧拉公式的代码实现** |
| 内容：单位圆图示、复平面几何、极坐标视角 | 内容：Python代码、FFT参数、旋转因子验证 |
| 目标：为什么 e^{iθ} = cos θ + i·sin θ | 目标：怎样用代码构造旋转因子 |

- **如果你已经学过 L06**：本课的"欧拉公式为什么成立"部分会是复习，焦点转向"如何用代码实现"。
- **如果你还没学过 L06**：完成本节后，建议回头看 L06 的单位圆图示和复平面几何部分，两边会相互印证，理解更深。

---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：欧拉公式与旋转因子手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：手算 `euler(θ)` 的三个特殊值（无需计算器）：
- euler(0) = ?
- euler(π/2) = ?
- euler(π) = ?

**问 2**：验证模长性质：`|euler(θ)|` 对任意 θ 都等于 ?  
（用勾股定理：√(cos²θ + sin²θ) = ?）

**问 3**：旋转因子 `twiddle(k=1, n=2, N=8)` = ?  
- 角度 = -2π·1·2/8 = ?（弧度）
- cos(?) + i·sin(?) = ?（精确值）

**问 4**：`twiddle(k=0, n=n, N=N)` 对任意 n、N 的值是 ?（为什么？）

**问 5**：N=8 时，`twiddle(1, n, 8)` 对 n=0,1,2,...,7 在单位圆上的排列规律是什么？

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：euler 特殊值
try:
    e0   = euler(0)
    e90  = euler(np.pi/2)
    e180 = euler(np.pi)
    assert np.isclose(e0,   1+0j,  atol=1e-12), f"euler(0) 应=1+0j，得到 {e0}"
    assert np.isclose(e90,  0+1j,  atol=1e-12), f"euler(π/2) 应=0+1j，得到 {e90}"
    assert np.isclose(e180, -1+0j, atol=1e-12), f"euler(π) 应=-1+0j，得到 {e180}"
    print(f"Q1 ✅  euler(0)={e0}  euler(π/2)={e90}  euler(π)={e180}")
except (NotImplementedError, TypeError):
    print("⬜ Q1：请先实现 euler()，再运行对答案格")

# 问2：模长恒=1
thetas = np.linspace(0, 2*np.pi, 1000)
mags = np.abs(np.cos(thetas) + 1j*np.sin(thetas))
assert np.allclose(mags, 1.0, atol=1e-12)
print(f"Q2 ✅  |euler(θ)| = 1.0 对所有 θ 成立（单位圆性质）")

# 问3：twiddle(1, 2, 8) = e^{-πi/2} = 0 - 1j
angle_312 = -2 * np.pi * 1 * 2 / 8  # = -π/2
expected_312 = np.cos(angle_312) + 1j*np.sin(angle_312)
assert np.isclose(expected_312, 0-1j, atol=1e-12), f"twiddle(1,2,8) 应=0-1j，得到 {expected_312}"
try:
    tw = twiddle(1, 2, 8)
    assert np.isclose(tw, 0-1j, atol=1e-12)
    print(f"Q3 ✅  twiddle(1,2,8) = e^{{-πi/2}} = {tw}")
except (NotImplementedError, TypeError):
    print("⬜ Q3：请先实现 twiddle()，再运行对答案格")

# 问4：twiddle(0, n, N) = e^0 = 1 对所有 n
try:
    for n_val in range(8):
        assert np.isclose(twiddle(0, n_val, 8), 1+0j, atol=1e-12)
    print(f"Q4 ✅  twiddle(0,n,N)=e^0=1 对所有 n（k=0 对应 DC 分量，旋转因子为1）")
except (NotImplementedError, TypeError):
    print("⬜ Q4：请先实现 twiddle()，再运行对答案格")

# 问5：N=8 的旋转因子均匀分布在单位圆
angles_8 = np.array([-2*np.pi*1*n/8 for n in range(8)])
tw_8 = np.cos(angles_8) + 1j*np.sin(angles_8)
assert np.allclose(np.abs(tw_8), 1.0, atol=1e-12)  # 都在单位圆
# 相邻相位差固定 = -π/4。注意 np.angle 输出被折叠到 (-π, π]，
# 差值需要重新折回同一区间再比较（否则在 ±π 跳变处会出现 +7π/4）。
diffs = np.diff(np.angle(tw_8))
diffs_wrapped = (diffs + np.pi) % (2 * np.pi) - np.pi
assert np.allclose(diffs_wrapped, -np.pi/4, atol=1e-12)
print(f"Q5 ✅  8个旋转因子均匀分布，每步转 -π/4={np.degrees(-np.pi/4):.1f}°")
print("\n🎉 欧拉公式白板挑战通过！旋转因子 e^{{-2πikn/N}} 是 FFT 的核心积木。")

## 补充：负角度在单位圆上的几何含义

在白板挑战中会遇到负角度（比如 -π/2）。**负号只代表旋转方向**：
- **正角度**（如 π/4）：逆时针旋转
- **负角度**（如 -π/4）：顺时针旋转

同一个点可以用多个角度表示：
- -π/2（顺时针转 90°）= 3π/2（逆时针转 270°）= 同一个点，位置在单位圆的下方 (0, -1)
- cos(-π/2) = cos(3π/2) = 0
- sin(-π/2) = sin(3π/2) = -1

**记住三角函数的对称性**：
- cos(-θ) = cos(θ)（余弦是偶函数）
- sin(-θ) = -sin(θ)（正弦是奇函数）

所以手算 twiddle(1, 2, 8) = euler(-π/2) 时：
- cos(-π/2) = cos(π/2) = 0 ✓
- sin(-π/2) = -sin(π/2) = -1 ✓
- 结果 = 0 + (-1)i = -1j ✓

In [ ]:
# ✏️ 本课自评
l35_review = {
    "euler_formula":          None,  # 记住 e^{iθ}=cosθ+i·sinθ？True/False
    "euler_implemented":      None,  # euler(theta) 实现并通过断言？True/False
    "twiddle_implemented":    None,  # twiddle(k,n,N) 实现并通过断言？True/False
    "unit_circle_property":   None,  # 理解 |e^{iθ}|=1（单位圆）？True/False
    "whiteboard_passed":      None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l35_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l35_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L35 全部通关！进入 L36：窗函数原理')

---

→ **下一课**　[L36 · 窗函数原理](L36_windows.ipynb)

> 下节课将学习 **窗函数原理**：矩形窗的旁瓣泄漏，Hann / Hamming / Blackman 对比。